# Card Embedding & Network Diagnostic

This notebook tests whether card embeddings differentiate between cards and whether card information flows through the network properly.

**Key Questions:**
1. Do different cards produce different embedding vectors?
2. Does the full network produce different outputs for different hands?
3. Where does information loss occur (if any)?
4. Is normalization squashing signal?
5. Are gradients flowing properly?

## Setup

In [1]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

# Add paths
sys.path.insert(0, '.')
sys.path.insert(0, '..')

from network.card_embedding import CardEmbedding
from network.model import DeepCFRModule, normalize
from utils.infoset_parser import parse_infoset_to_network_input

# Optional: Load a trained model for comparison
MODEL_PATH = "../output/models/fix_test.pt"  # Change to your model

print("Setup complete!")

Setup complete!


---
## Test 1: Card Embedding Differentiation

Check if different cards produce different embedding vectors.

In [2]:
print("="*60)
print("TEST 1: Card Embedding Differentiation")
print("="*60)

# Create fresh embedding layer
embedding = CardEmbedding(dim=256)

# Card encoding: rank * 10 + suit
# Ace of spades = 14*10 + 0 = 140
# 2 of spades = 2*10 + 0 = 20
# King of hearts = 13*10 + 1 = 131

test_cards = [
    ("Ace of spades", 140),
    ("2 of spades", 20),
    ("King of hearts", 131),
    ("7 of diamonds", 72),
    ("Ace of hearts", 141),  # Same rank as Ace of spades
]

embeddings = {}
for name, code in test_cards:
    card_tensor = torch.tensor([[code]])
    emb = embedding(card_tensor)
    embeddings[name] = emb
    print(f"{name} (code={code}): mean={emb.mean().item():.4f}, std={emb.std().item():.4f}, norm={emb.norm().item():.4f}")

print("\nPairwise differences (should be > 0):")
print("-" * 40)

# Compare Ace vs 2 (very different ranks)
diff_ace_2 = (embeddings["Ace of spades"] - embeddings["2 of spades"]).abs().mean().item()
print(f"Ace vs 2 (different ranks): {diff_ace_2:.6f}")

# Compare Ace spades vs Ace hearts (same rank, different suit)
diff_ace_suits = (embeddings["Ace of spades"] - embeddings["Ace of hearts"]).abs().mean().item()
print(f"As vs Ah (same rank, diff suit): {diff_ace_suits:.6f}")

# Compare King vs 7 
diff_king_7 = (embeddings["King of hearts"] - embeddings["7 of diamonds"]).abs().mean().item()
print(f"King vs 7: {diff_king_7:.6f}")

# Assessment
print("\n" + "="*60)
if diff_ace_2 > 0.01:
    print("PASS: Card embeddings differentiate between different ranks")
else:
    print("FAIL: Card embeddings NOT differentiating - this is a problem!")

if diff_ace_suits > 0.001:
    print("PASS: Card embeddings differentiate between different suits")
else:
    print("WARN: Suit differentiation is weak (may be OK)")

TEST 1: Card Embedding Differentiation
Ace of spades (code=140): mean=-0.0744, std=1.7991, norm=28.7547
2 of spades (code=20): mean=-0.0272, std=1.7004, norm=27.1563
King of hearts (code=131): mean=0.0821, std=1.7219, norm=27.5286
7 of diamonds (code=72): mean=0.0971, std=1.8002, norm=28.7886
Ace of hearts (code=141): mean=0.0925, std=1.7428, norm=27.8698

Pairwise differences (should be > 0):
----------------------------------------
Ace vs 2 (different ranks): 1.653682
As vs Ah (same rank, diff suit): 1.590659
King vs 7: 1.997020

PASS: Card embeddings differentiate between different ranks
PASS: Card embeddings differentiate between different suits


---
## Test 2: Network Output Differentiation (Fresh Network)

Check if a freshly initialized network produces different outputs for different hands.

In [3]:
print("="*60)
print("TEST 2: Network Output Differentiation (FRESH Network)")
print("="*60)

# Create fresh network (not trained)
fresh_network = DeepCFRModule(
    nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256
)
fresh_network.eval()

# Test hands - preflop (no board)
test_infosets = [
    ("AA-x (premium)", "S0|H:14s0,14s1,8s2|B:|A:"),
    ("KK-x (premium)", "S0|H:13s0,13s1,5s2|B:|A:"),
    ("2-3-4 (garbage)", "S0|H:4s0,3s1,2s2|B:|A:"),
    ("7-5-3 (low)", "S0|H:7s0,5s1,3s2|B:|A:"),
]

outputs = {}
print("\nRaw network outputs (logits) for different hands:")
print("-" * 60)

for desc, infoset in test_infosets:
    cc, ah = parse_infoset_to_network_input(infoset)
    with torch.no_grad():
        out = fresh_network(cc, ah)
    outputs[desc] = out[0]
    print(f"{desc}:")
    print(f"  Logits: [{', '.join([f'{x:.4f}' for x in out[0].tolist()])}]")
    print(f"  Mean: {out[0].mean().item():.6f}, Std: {out[0].std().item():.6f}")
    print()

# Compare outputs
print("\nPairwise output differences:")
print("-" * 40)

diff_aa_garbage = (outputs["AA-x (premium)"] - outputs["2-3-4 (garbage)"]).abs().mean().item()
diff_aa_kk = (outputs["AA-x (premium)"] - outputs["KK-x (premium)"]).abs().mean().item()
diff_garbage_low = (outputs["2-3-4 (garbage)"] - outputs["7-5-3 (low)"]).abs().mean().item()

print(f"AA vs 2-3-4: {diff_aa_garbage:.6f}")
print(f"AA vs KK: {diff_aa_kk:.6f}")
print(f"2-3-4 vs 7-5-3: {diff_garbage_low:.6f}")

# Assessment
print("\n" + "="*60)
if diff_aa_garbage > 0.001:
    print("PASS: Fresh network produces different outputs for different hands")
else:
    print("FAIL: Fresh network outputs are identical - architecture problem!")
    
# Check if outputs are near zero (as expected from initialization)
max_logit = max(abs(outputs["AA-x (premium)"].max().item()), abs(outputs["AA-x (premium)"].min().item()))
if max_logit < 0.1:
    print(f"INFO: Logits are near zero ({max_logit:.4f}) - this is expected from zero initialization")
else:
    print(f"WARN: Logits are not near zero ({max_logit:.4f}) - unexpected for fresh network")

TEST 2: Network Output Differentiation (FRESH Network)

Raw network outputs (logits) for different hands:
------------------------------------------------------------
AA-x (premium):
  Logits: [-0.0162, -0.0052, 0.0181, 0.0090, -0.0135, 0.0151, -0.0179, 0.0025, 0.0158]
  Mean: 0.000855, Std: 0.014487

KK-x (premium):
  Logits: [-0.0125, 0.0039, 0.0125, 0.0018, -0.0055, 0.0251, -0.0040, 0.0076, 0.0186]
  Mean: 0.005275, Std: 0.012057

2-3-4 (garbage):
  Logits: [-0.0144, 0.0035, 0.0147, 0.0090, -0.0122, 0.0143, -0.0228, -0.0038, 0.0166]
  Mean: 0.000540, Std: 0.014478

7-5-3 (low):
  Logits: [-0.0232, -0.0066, 0.0175, 0.0084, -0.0066, 0.0174, -0.0127, 0.0056, -0.0026]
  Mean: -0.000314, Std: 0.013715


Pairwise output differences:
----------------------------------------
AA vs 2-3-4: 0.003143
AA vs KK: 0.007260
2-3-4 vs 7-5-3: 0.007770

PASS: Fresh network produces different outputs for different hands
INFO: Logits are near zero (0.0181) - this is expected from zero initialization


---
## Test 3: Network Output Differentiation (Trained Network)

Check if the trained network produces different outputs for different hands.

In [4]:
print("="*60)
print("TEST 3: Network Output Differentiation (TRAINED Network)")
print("="*60)

# Load trained model
try:
    model_data = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
    network_dim = model_data.get('network_dim', 256)
    iterations = model_data.get('iterations', '?')
    
    trained_network = DeepCFRModule(
        nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=network_dim
    )
    trained_network.load_state_dict(model_data['strategy_network_state_dict'])
    trained_network.eval()
    print(f"Loaded model: {MODEL_PATH} ({iterations} iterations)")
except Exception as e:
    print(f"Could not load model: {e}")
    trained_network = None

if trained_network:
    outputs_trained = {}
    print("\nRaw network outputs (logits) for different hands:")
    print("-" * 60)
    
    for desc, infoset in test_infosets:
        cc, ah = parse_infoset_to_network_input(infoset)
        with torch.no_grad():
            out = trained_network(cc, ah)
        outputs_trained[desc] = out[0]
        print(f"{desc}:")
        print(f"  Logits: [{', '.join([f'{x:.4f}' for x in out[0].tolist()])}]")
        print(f"  Mean: {out[0].mean().item():.6f}, Std: {out[0].std().item():.6f}")
        print()
    
    # Compare outputs
    print("\nPairwise output differences:")
    print("-" * 40)
    
    diff_aa_garbage_t = (outputs_trained["AA-x (premium)"] - outputs_trained["2-3-4 (garbage)"]).abs().mean().item()
    diff_aa_kk_t = (outputs_trained["AA-x (premium)"] - outputs_trained["KK-x (premium)"]).abs().mean().item()
    
    print(f"AA vs 2-3-4: {diff_aa_garbage_t:.6f}")
    print(f"AA vs KK: {diff_aa_kk_t:.6f}")
    
    # Assessment
    print("\n" + "="*60)
    if diff_aa_garbage_t > 0.01:
        print("PASS: Trained network differentiates hands")
    else:
        print("FAIL: Trained network outputs are nearly identical!")
        print("      This suggests card information is NOT being learned.")

TEST 3: Network Output Differentiation (TRAINED Network)
Loaded model: ../output/models/fix_test.pt (20 iterations)

Raw network outputs (logits) for different hands:
------------------------------------------------------------
AA-x (premium):
  Logits: [1.0197, 0.0000, -0.0010, 0.1143, -0.0243, 0.7424, 0.0310, 0.0210, 0.0160]
  Mean: 0.213223, Std: 0.386817

KK-x (premium):
  Logits: [1.0178, 0.0000, -0.0013, 0.1134, -0.0248, 0.7416, 0.0297, 0.0220, 0.0179]
  Mean: 0.212912, Std: 0.386161

2-3-4 (garbage):
  Logits: [1.0178, 0.0000, 0.0001, 0.1180, -0.0241, 0.7264, 0.0362, 0.0243, 0.0204]
  Mean: 0.213233, Std: 0.382630

7-5-3 (low):
  Logits: [1.0174, 0.0000, 0.0001, 0.1173, -0.0230, 0.7226, 0.0383, 0.0250, 0.0211]
  Mean: 0.213208, Std: 0.381595


Pairwise output differences:
----------------------------------------
AA vs 2-3-4: 0.003974
AA vs KK: 0.000972

FAIL: Trained network outputs are nearly identical!
      This suggests card information is NOT being learned.


---
## Test 4: Feature Flow Check

Check if card features survive through the network layers using hooks.

In [5]:
print("="*60)
print("TEST 4: Feature Flow Through Network Layers")
print("="*60)

# Create fresh network for testing
test_net = DeepCFRModule(
    nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256
)
test_net.eval()

# Storage for intermediate activations
activations = {}

def make_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks on key layers
hooks = [
    test_net.hand_layer1.register_forward_hook(make_hook('hand_layer1')),
    test_net.hand_layer2.register_forward_hook(make_hook('hand_layer2')),
    test_net.hand_layer3.register_forward_hook(make_hook('hand_layer3')),
    test_net.comb_layer1.register_forward_hook(make_hook('comb_layer1')),
    test_net.comb_layer2.register_forward_hook(make_hook('comb_layer2')),
    test_net.comb_layer3.register_forward_hook(make_hook('comb_layer3')),
    test_net.action_head.register_forward_hook(make_hook('action_head')),
]

# Run two different hands through
print("\nComparing activations for AA-x vs 2-3-4:")
print("-" * 60)

cc1, ah1 = parse_infoset_to_network_input("S0|H:14s0,14s1,8s2|B:|A:")  # AA-x
cc2, ah2 = parse_infoset_to_network_input("S0|H:4s0,3s1,2s2|B:|A:")    # 2-3-4

with torch.no_grad():
    out1 = test_net(cc1, ah1)
    act1 = {k: v.clone() for k, v in activations.items()}
    
    out2 = test_net(cc2, ah2)
    act2 = {k: v.clone() for k, v in activations.items()}

# Compare activations at each layer
print(f"{'Layer':<15} {'Mean Diff':<12} {'Max Diff':<12} {'AA-x Norm':<12} {'2-3-4 Norm':<12}")
print("-" * 65)

for layer_name in ['hand_layer1', 'hand_layer2', 'hand_layer3', 'comb_layer1', 'comb_layer2', 'comb_layer3', 'action_head']:
    diff = (act1[layer_name] - act2[layer_name]).abs()
    mean_diff = diff.mean().item()
    max_diff = diff.max().item()
    norm1 = act1[layer_name].norm().item()
    norm2 = act2[layer_name].norm().item()
    print(f"{layer_name:<15} {mean_diff:<12.6f} {max_diff:<12.6f} {norm1:<12.4f} {norm2:<12.4f}")

# Remove hooks
for h in hooks:
    h.remove()

# Assessment
print("\n" + "="*60)
action_diff = (act1['action_head'] - act2['action_head']).abs().mean().item()
if action_diff > 0.001:
    print(f"PASS: Differences propagate to output layer (diff={action_diff:.6f})")
else:
    print(f"FAIL: Differences do NOT reach output layer (diff={action_diff:.6f})")
    print("      Information is being lost somewhere in the network!")

TEST 4: Feature Flow Through Network Layers

Comparing activations for AA-x vs 2-3-4:
------------------------------------------------------------
Layer           Mean Diff    Max Diff     AA-x Norm    2-3-4 Norm  
-----------------------------------------------------------------
hand_layer1     0.945166     3.928328     15.6712      15.7475     
hand_layer2     0.282806     0.939764     5.8122       5.6641      
hand_layer3     0.098087     0.356742     2.4996       2.4333      
comb_layer1     0.021796     0.084932     0.6636       0.6826      
comb_layer2     0.008267     0.030163     0.6603       0.6556      
comb_layer3     0.008177     0.029203     0.7895       0.8152      
action_head     0.003333     0.009002     0.0379       0.0421      

PASS: Differences propagate to output layer (diff=0.003333)


---
## Test 5: Normalization Impact

Check if the normalize() function is squashing differences.

In [6]:
print("="*60)
print("TEST 5: Normalization Impact")
print("="*60)

# Test the normalize function directly
print("\nTesting normalize() function:")
print("-" * 40)

# Create test vectors with different characteristics
vec1 = torch.randn(1, 256) * 10  # Large variance
vec2 = torch.randn(1, 256) * 0.01  # Small variance
vec3 = torch.ones(1, 256) * 5 + torch.randn(1, 256) * 0.001  # Nearly constant

for name, vec in [('Large var', vec1), ('Small var', vec2), ('Nearly constant', vec3)]:
    norm_vec = normalize(vec)
    print(f"{name}:")
    print(f"  Before: mean={vec.mean().item():.4f}, std={vec.std().item():.4f}")
    print(f"  After:  mean={norm_vec.mean().item():.4f}, std={norm_vec.std().item():.4f}")
    print()

# Test with actual network features
print("\nComparing features before/after normalize (from comb_layer3):")
print("-" * 60)

# Get features just before normalize
test_net2 = DeepCFRModule(
    nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256
)
test_net2.eval()

pre_norm_features = {}

def capture_pre_norm(module, input, output):
    pre_norm_features['comb3_out'] = output.detach().clone()

hook = test_net2.comb_layer3.register_forward_hook(capture_pre_norm)

cc1, ah1 = parse_infoset_to_network_input("S0|H:14s0,14s1,8s2|B:|A:")
cc2, ah2 = parse_infoset_to_network_input("S0|H:4s0,3s1,2s2|B:|A:")

with torch.no_grad():
    _ = test_net2(cc1, ah1)
    feat1_pre = pre_norm_features['comb3_out'].clone()
    
    _ = test_net2(cc2, ah2)
    feat2_pre = pre_norm_features['comb3_out'].clone()

hook.remove()

# Apply normalize manually
feat1_post = normalize(feat1_pre)
feat2_post = normalize(feat2_pre)

diff_pre = (feat1_pre - feat2_pre).abs().mean().item()
diff_post = (feat1_post - feat2_post).abs().mean().item()

print(f"Difference BEFORE normalize: {diff_pre:.6f}")
print(f"Difference AFTER normalize:  {diff_post:.6f}")
print(f"Ratio (post/pre): {diff_post/diff_pre:.4f}" if diff_pre > 0 else "Cannot compute ratio")

# Assessment
print("\n" + "="*60)
if diff_post < diff_pre * 0.1:
    print("WARN: Normalize is significantly reducing differences!")
    print("      Consider removing or modifying normalize()")
else:
    print("OK: Normalize preserves relative differences")

TEST 5: Normalization Impact

Testing normalize() function:
----------------------------------------
Large var:
  Before: mean=0.1881, std=9.9895
  After:  mean=-0.0000, std=1.0000

Small var:
  Before: mean=0.0009, std=0.0102
  After:  mean=0.0000, std=0.9999

Nearly constant:
  Before: mean=5.0000, std=0.0009
  After:  mean=-0.0001, std=0.9989


Comparing features before/after normalize (from comb_layer3):
------------------------------------------------------------
Difference BEFORE normalize: 0.007419
Difference AFTER normalize:  0.170044
Ratio (post/pre): 22.9204

OK: Normalize preserves relative differences


---
## Test 6: Training Gradient Flow

Verify gradients are actually flowing to embedding layers.

In [7]:
print("="*60)
print("TEST 6: Training Gradient Flow")
print("="*60)

# Create network for gradient testing
grad_net = DeepCFRModule(
    nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256
)
grad_net.train()  # Set to training mode

# Forward pass
cc, ah = parse_infoset_to_network_input("S0|H:14s0,14s1,8s2|B:|A:")
output = grad_net(cc, ah)

# Create dummy target and loss
target = torch.zeros_like(output)
target[0, 6] = 1.0  # Target: RAISE_SMALL

loss = F.mse_loss(output, target)
loss.backward()

# Check gradients at key layers
print("\nGradient magnitudes at key layers:")
print("-" * 60)

layers_to_check = [
    ('hand_embeddings[0].card.weight', grad_net.hand_embeddings[0].card.weight),
    ('hand_embeddings[0].rank.weight', grad_net.hand_embeddings[0].rank.weight),
    ('hand_layer1.weight', grad_net.hand_layer1.weight),
    ('hand_layer2.weight', grad_net.hand_layer2.weight),
    ('comb_layer1.weight', grad_net.comb_layer1.weight),
    ('comb_layer3.weight', grad_net.comb_layer3.weight),
    ('action_head.weight', grad_net.action_head.weight),
    ('action_head.bias', grad_net.action_head.bias),
]

print(f"{'Layer':<35} {'Grad Mean':<15} {'Grad Max':<15} {'Has Grad'}")
print("-" * 80)

all_grads_ok = True
for name, param in layers_to_check:
    if param.grad is not None:
        grad_mean = param.grad.abs().mean().item()
        grad_max = param.grad.abs().max().item()
        has_grad = "Yes"
        if grad_mean == 0:
            all_grads_ok = False
    else:
        grad_mean = 0
        grad_max = 0
        has_grad = "NO!"
        all_grads_ok = False
    print(f"{name:<35} {grad_mean:<15.8f} {grad_max:<15.8f} {has_grad}")

# Assessment
print("\n" + "="*60)
if all_grads_ok:
    print("PASS: Gradients flow to all layers including embeddings")
else:
    print("FAIL: Some layers have zero or no gradients!")
    print("      Check for gradient blocking operations.")

TEST 6: Training Gradient Flow

Gradient magnitudes at key layers:
------------------------------------------------------------
Layer                               Grad Mean       Grad Max        Has Grad
--------------------------------------------------------------------------------
hand_embeddings[0].card.weight      0.00000019      0.00012914      Yes
hand_embeddings[0].rank.weight      0.00000190      0.00012914      Yes
hand_layer1.weight                  0.00009684      0.00256173      Yes
hand_layer2.weight                  0.00007117      0.00333786      Yes
comb_layer1.weight                  0.00007068      0.00498437      Yes
comb_layer3.weight                  0.00007118      0.00131995      Yes
action_head.weight                  0.02100049      1.00640595      Yes
action_head.bias                    0.02664164      0.22157712      Yes

PASS: Gradients flow to all layers including embeddings


---
## Summary & Recommendations

In [8]:
print("="*60)
print("DIAGNOSTIC SUMMARY")
print("="*60)
print()
print("Run all cells above and check for FAIL/WARN messages.")
print()
print("Common issues and fixes:")
print("-" * 60)
print()
print("1. EMBEDDINGS NOT DIFFERENTIATING:")
print("   - Check CardEmbedding initialization")
print("   - Verify card encoding (rank*10 + suit)")
print()
print("2. FRESH NETWORK OUTPUTS IDENTICAL:")
print("   - Architecture problem - check layer connections")
print("   - Check if ReLU is killing all signal")
print()
print("3. TRAINED NETWORK OUTPUTS IDENTICAL:")
print("   - Not enough training iterations")
print("   - Learning rate too low")
print("   - Training targets don't differentiate hands")
print()
print("4. INFORMATION LOST IN LAYERS:")
print("   - Check where differences drop to near-zero")
print("   - May need to remove normalize() or adjust it")
print()
print("5. NORMALIZE SQUASHING SIGNAL:")
print("   - Try removing normalize() call in model.py")
print("   - Or use LayerNorm instead")
print()
print("6. GRADIENTS NOT FLOWING:")
print("   - Check for operations that block gradients")
print("   - Verify loss function is correct")

DIAGNOSTIC SUMMARY

Run all cells above and check for FAIL/WARN messages.

Common issues and fixes:
------------------------------------------------------------

1. EMBEDDINGS NOT DIFFERENTIATING:
   - Check CardEmbedding initialization
   - Verify card encoding (rank*10 + suit)

2. FRESH NETWORK OUTPUTS IDENTICAL:
   - Architecture problem - check layer connections
   - Check if ReLU is killing all signal

3. TRAINED NETWORK OUTPUTS IDENTICAL:
   - Not enough training iterations
   - Learning rate too low
   - Training targets don't differentiate hands

4. INFORMATION LOST IN LAYERS:
   - Check where differences drop to near-zero
   - May need to remove normalize() or adjust it

5. NORMALIZE SQUASHING SIGNAL:
   - Try removing normalize() call in model.py
   - Or use LayerNorm instead

6. GRADIENTS NOT FLOWING:
   - Check for operations that block gradients
   - Verify loss function is correct
